# Devoir 2 — Système RAG pour le Code de la Route (Loi 52.05)


Ce notebook implémente un système **Retrieval-Augmented Generation (RAG)** complet pour répondre à des questions juridiques en langage naturel à partir du Code de la Route marocain (Loi 52.05).

## Architecture RAG
```
Question utilisateur
       │
       ▼
  [Embeddings]  ←─ Modèle : paraphrase-multilingual-MiniLM-L12-v2
       │
       ▼
  [FAISS Index] ─→ Top-k documents pertinents
       │
       ▼
  [Prompt]      ─→ contexte + question
       │
       ▼
   [LLM]        ─→ Réponse contextualisée + références

```


**Réalisée par ACHARF EL BOUMASHOULI**

**Master IASD 2026 | Pr. I. BENABDELOUAHAB**

## 0. Installation des dépendances

In [1]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate \
              pdfplumber langchain-text-splitters gradio openai groq pdfminer.six
!pip install -q pytesseract pdf2image
!apt-get update -qq
!apt-get install -y poppler-utils tesseract-ocr tesseract-ocr-ara -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 0.1 Upload du PDF

In [2]:
from google.colab import files
uploaded = files.upload()

# Renommer le fichier uploadé en MA52_05.pdf
import os
for filename in uploaded.keys():
    if filename.endswith('.pdf'):
        os.rename(filename, 'MA52_05.pdf')
        print(f"✅ Fichier renommé: MA52_05.pdf")

## 1. Extraction du texte avec OCR amélioré

In [3]:
from pdf2image import convert_from_path
import pytesseract
import re
import pandas as pd

print("🔄 Conversion du PDF en images...")
images = convert_from_path("MA52_05.pdf", dpi=300)
print(f"📄 {len(images)} pages extraites")

# Configuration OCR améliorée pour l'arabe
custom_config = r'--psm 6'

print("🔄 Extraction du texte avec OCR (cela peut prendre quelques minutes)...")
full_text_raw = ""
for i, img in enumerate(images):
    text = pytesseract.image_to_string(img, lang='ara', config=custom_config)
    full_text_raw += text + "\n"

print(f"✅ Texte extrait: {len(full_text_raw)} caractères")

# Aperçu du texte extrait
print("\n📝 Aperçu du texte extrait:")
print(full_text_raw[:1500])

🔄 Conversion du PDF en images...
📄 126 pages extraites
🔄 Extraction du texte avec OCR (cela peut prendre quelques minutes)...
✅ Texte extrait: 200542 caractères

📝 Aperçu du texte extrait:
المملكة المغربية 2 أمانة العامة :

| ل | م شمف

القانوز رقم 32.05 المتعلق

بمدونة السير عدر الضرؤى
كما وقع تغييرق وتتميمة
صيغة موئمرة متازيخ
يليو 2024

يراب +7 حك ار ا 2 ا 1115 سخ كار ا
2[ لكلو ايا سدم رمت ويل
1
الحا اي :ا السلا

القانون رقم 52.05
المتعلق بمدونة السير على الطرق الصادر بتنفيذه الظبير الشريف رقم 1.10.07
بتاريخ 26 من صفر 1431 (11 فبراير 2010),
كما وقع تغييره وتتميمه
(ج.ر عدد 5824 بتاريخ 8 ربيع الآخر 25(1431 مارس 2010). ص : 2168)
الكتاب الأول
شروط السير على الطريق العمومية
القسم الأول
رخصة السياقة
الباب الأول
إلزامية رخصة السياقة
المادة 1
لا يجوز لأي شخص أن يسوق مركبة ذات محرك أو مجموعة مركبات على الطريق العمومية
ما لم يكن حاصلا على رخصة للسياقة سارية الصلاحية ومسلمة من قبل الإدارة. تناسب صنف
المركبة أو مجموعة المركبات التي يسوقها.
المادة 2
الصلاحية ؛
سارية الصلاحية. لكن فقطء خلال مدة أ

In [4]:
import re
print(re.findall(r'المادة', full_text_raw)[:10])

['المادة', 'المادة', 'المادة', 'المادة', 'المادة', 'المادة', 'المادة', 'المادة', 'المادة', 'المادة']


## 2. Parsing et structuration des articles

In [5]:
import re
import pandas as pd

def clean_arabic_text(text):
    """Nettoie et normalise le texte arabe."""
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', text)  # control chars
    text = re.sub(r'\f', ' ', text)                  # page breaks
    text = re.sub(r'\s+', ' ', text)                 # normalize spaces
    text = re.sub(r'[-ÿ]', '', text)                # weird latin chars
    return text.strip()


def parse_articles_fixed(full_text):
    """
    Version robuste pour parser les articles arabes.
    """
    records = []

    # Nettoyage global AVANT parsing (important)
    full_text = clean_arabic_text(full_text)

    # Pattern plus robuste
    pattern = r'(المادة\s*\d+|املادة\s*\d+)'

    parts = re.split(pattern, full_text)

    i = 1
    seen_numbers = set()

    while i < len(parts):
        header = parts[i].strip()
        body = parts[i + 1].strip() if i + 1 < len(parts) else ""

        # Extract number
        num_match = re.search(r'(\d+)', header)
        if not num_match:
            i += 2
            continue

        num = int(num_match.group(1))

        if num in seen_numbers:
            i += 2
            continue
        seen_numbers.add(num)

        if len(body) > 30:
            records.append({
                "article_num": num,
                "article": f"المادة {num}",
                "texte": body[:2000]
            })

        i += 2

    # Sort
    records.sort(key=lambda x: x["article_num"])

    return pd.DataFrame(records)


print("🔄 Parsing des articles...")
df = parse_articles_fixed(full_text_raw)
print(f"✅ {len(df)} articles extraits")
df.head(10)

🔄 Parsing des articles...
✅ 312 articles extraits


,article_num,article,texte
0,1,المادة 1,لا يجوز لأي شخص أن يسوق مركبة ذات محرك أو مجمو...
1,2,المادة 2,الصلاحية ؛سارية الصلاحية. لكن فقطء خلال مدة أق...
2,3,المادة 3,يجب على السائقين الحاصلين على رخصة سياقة مسلمة...
3,4,المادة 4,في حالة السير الدولي ووفقا للاتفاقية الدولية ل...
4,5,المادة 5,(غيرت وتممت موجب امادة الأولى من القانون رقم 1...
5,6,المادة 6,لا يجوز لأي كان سياقة مركبة فلاحية ذات محرك أو...
6,7,المادة 7,(غيرت وتممت موجب امادة الأولى من القانون رقم 1...
7,8,المادة 8,(غيرت وتممت موجب امادة الأولى من القانون رقم 1...
8,9,المادة 9,يجب الإدلاء برخصة السياقة أو بالوثيقة التي تحل...
9,10,المادة 10,أسفله شريطة استيفائهم الشروط المحددة في البندي...


## 3. Sauvegarde du CSV et découpage en chunks

In [6]:
# Sauvegarde CSV
CSV_PATH = "code_de_la_route_52_05.csv"
df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
print(f"✅ CSV sauvegardé : {CSV_PATH}")

# Chunking optimisé pour l'arabe
def chunk_arabic_text(text, article_id, article_label, chunk_size=400, overlap=80):
    """
    Découpe un texte arabe en chunks avec chevauchement.
    Utilise la longueur en caractères plutôt qu'en mots pour l'arabe.
    """
    chunks = []
    chunk_idx = 0
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)

        # Essayer de couper à la fin d'une phrase
        if end < text_len:
            # Chercher . أو ; أو : أو ؟ أو !
            for sep in ['\.', ';', ':', '؟', '!', '\n']:
                last_sep = text.rfind(sep, start, end)
                if last_sep > start + chunk_size // 2:
                    end = last_sep + 1
                    break

        chunk_text = text[start:end].strip()
        if chunk_text:
            chunks.append({
                "chunk_id": f"{article_id}_{chunk_idx}",
                "article_num": article_id,
                "article": article_label,
                "texte": chunk_text
            })

        start = end - overlap if end < text_len else text_len
        chunk_idx += 1

    return chunks

all_chunks = []
for _, row in df.iterrows():
    all_chunks.extend(chunk_arabic_text(row["texte"], row["article_num"], row["article"]))

df_chunks = pd.DataFrame(all_chunks)
print(f"✅ {len(df_chunks)} chunks créés depuis {len(df)} articles")
df_chunks.head()

✅ CSV sauvegardé : code_de_la_route_52_05.csv
✅ 515 chunks créés depuis 312 articles


<>:23: SyntaxWarning: invalid escape sequence '\.'
<>:23: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_11697/2452061099.py:23: SyntaxWarning: invalid escape sequence '\.'
  for sep in ['\.', ';', ':', '؟', '!', '\n']:


,chunk_id,article_num,article,texte
0,1_0,1,المادة 1,لا يجوز لأي شخص أن يسوق مركبة ذات محرك أو مجمو...
1,2_0,2,المادة 2,الصلاحية ؛سارية الصلاحية. لكن فقطء خلال مدة أق...
2,3_0,3,المادة 3,يجب على السائقين الحاصلين على رخصة سياقة مسلمة...
3,3_1,3,المادة 3,السياقة. تبديل سنداتهم مقابل رخصة سياقة مغربية...
4,4_0,4,المادة 4,في حالة السير الدولي ووفقا للاتفاقية الدولية ل...


## 4. Génération des embeddings et indexation FAISS

In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Modèle amélioré pour l'arabe juridique
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print(f"🔄 Chargement du modèle : {EMBEDDING_MODEL}")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

texts = df_chunks["texte"].tolist()

print(f"🔄 Encodage de {len(texts)} chunks...")
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True
)

dimension = embeddings.shape[1]

# Index FAISS pour similarité cosinus
index = faiss.IndexFlatIP(dimension)
index.add(np.array(embeddings, dtype="float32"))

print(f"✅ Index FAISS construit : {index.ntotal} vecteurs")

# Sauvegarder l'index
faiss.write_index(index, "faiss_index.bin")
print("✅ Index sauvegardé")

🔄 Chargement du modèle : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Encodage de 515 chunks...


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

✅ Index FAISS construit : 515 vecteurs
✅ Index sauvegardé


## 5. Fonction de recherche améliorée (Retriever)

In [8]:
def retrieve(query, k=5, min_score=0.3):
    """
    Recherche les chunks les plus pertinents.
    """
    query_vec = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1 or score < min_score:
            continue

        row = df_chunks.iloc[idx]
        results.append({
            "texte": row["texte"],
            "article": row["article"],
            "article_num": int(row["article_num"]),
            "similarity": float(score),
            "chunk_id": row["chunk_id"]
        })

    return results

def is_out_of_domain(query, threshold=0.25):
    """
    Détecte si une question est hors domaine.
    Retourne True si la meilleure similarité est faible.
    """
    results = retrieve(query, k=1)
    if not results:
        return True
    return results[0]["similarity"] < threshold

# Test du retriever
test_queries = [
    "ما هي شروط الحصول على رخصة السياقة؟",
    "ما هو الحد الأقصى للسرعة في المدينة؟",
    "ما هي عقوبة القيادة تحت تأثير الكحول؟"
]

print("🔍 Test du retriever:")
for q in test_queries:
    results = retrieve(q, k=3)
    print(f"\n📝 Question: {q}")
    if results:
        for r in results:
            print(f"   → {r['article']} (sim={r['similarity']:.3f})")
            print(f"     {r['texte'][:100]}...")
    else:
        print("   ❌ Aucun résultat trouvé")

🔍 Test du retriever:

📝 Question: ما هي شروط الحصول على رخصة السياقة؟
   → المادة 152 (sim=0.819)
     يعاقب بغرامة من ألفي (2.000) درهم إلى ثمانية آللااف (8.000) درهم. كل شخص صدر في حقه1 - لم يودع رخصة ...
   → المادة 37 (sim=0.778)
     016). ص: 5867)يجب أن يتضمن الحامل المحررة فيه رخصة السياقة . على الخصوص ما يلي :- البيانات المتعلقة ...
   → المادة 132 (sim=0.772)
     تبلغ المعلومات المتعلقة بوجود رخصة السياقة وصنفها وصلاحيتها وبهوية صاحهاء بناء على طلهمإلى :1 - محام...

📝 Question: ما هو الحد الأقصى للسرعة في المدينة؟
   → المادة 185 (sim=0.591)
     مخالفة من الدرجةالثانية.تعتبر مخالفة من الدرجة الثانية إحدى المخالفات التالية :1 -تجاوز السرعة القصو...
   → المادة 184 (sim=0.579)
     ة من سبعمائة (700) إلى ألفتعتبر مخالفة من الدرجة الأولى إحدى المخالفات التالية :1 - تجاوز السرعة الق...
   → المادة 207 (sim=0.506)
     عند | ارعدم كفاية الرؤية وذلك دون إنارة أو دون تشوير.8 تجاوز السرعة المسموح بها بما يفوق 20 كيلومترا...

📝 Question: ما هي عقوبة القيادة تحت تأثير الكحول؟
   → الم

## 6. Intégration des LLM et pipeline RAG

In [24]:
import os
from groq import Groq

# Configuration Groq
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "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")
groq_client = Groq(api_key=GROQ_API_KEY)

LLM_MODELS = {
    "LLaMA-3.3-70B": "llama-3.3-70b-versatile",
    "LLaMA-3.1-8B": "llama-3.1-8b-instant"
}

def build_prompt(query, docs):
    """Construit le prompt pour le LLM."""
    context_parts = []
    for i, doc in enumerate(docs, 1):
        context_parts.append(f"[{doc['article']}]:\n{doc['texte'][:500]}")
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""أنت مساعد قانوني متخصص في قانون السير المغربي (القانون 52.05).

تعليمات:
- أجب اعتماداً على السياق فقط
- إذا كانت الإجابة جزئية، قدم أفضل إجابة ممكنة
- لا تقل "لا توجد معلومات" إلا إذا كان السياق غير مرتبط تماماً
- اذكر رقم المادة (المادة X)
- أجب بالعربية الفصحى
- اذكر المصدر بهذا الشكل: (المادة X)
السياق:
{context}

السؤال: {query}

الإجابة:"""

    return prompt

def generate_answer(query, model_key="LLaMA-3.1-8B", k=5):
    """Pipeline RAG complet."""

    # Vérification hors domaine
    out_of_domain = is_out_of_domain(query)

    # Recherche des documents pertinents
    docs = retrieve(query, k=k, min_score=0.25)

    if not docs:
        return {
            "answer": "❌ لم يتم العثور على معلومات كافية في قانون السير.",
            "model": model_key,
            "sources": [],
            "out_of_domain": out_of_domain
        }

    # Construction du prompt
    prompt = build_prompt(query, docs)

    if out_of_domain:
        prompt = "⚠️ NOTE: Cette question peut être hors du domaine du code de la route.\n\n" + prompt

    # Appel au LLM
    model_id = LLM_MODELS.get(model_key, "llama-3.1-8b-instant")

    try:
        response = groq_client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
            temperature=0.2
        )
        answer = response.choices[0].message.content.strip()
    except Exception as e:
        answer = f"❌ Erreur LLM: {str(e)}"

    return {
        "answer": answer,
        "model": model_key,
        "sources": [{"article": d["article"], "texte": d["texte"][:200]} for d in docs[:3]],
        "out_of_domain": out_of_domain
    }

# Test
print("🔄 Test du pipeline RAG:")
test_query = "ما هي شروط الحصول على رخصة السياقة؟"
result = generate_answer(test_query)
print(f"\n📝 Question: {test_query}")
print(f"🤖 Réponse: {result['answer'][:300]}")
print(f"📚 Sources: {[s['article'] for s in result['sources']]}")

🔄 Test du pipeline RAG:

📝 Question: ما هي شروط الحصول على رخصة السياقة؟
🤖 Réponse: حسناً. الإجابة هي: (المادة 251)

يجب أن يكون الحاصل على رخصة السياقة حاصلا على:

- رخصة السياقة بعد انتهاء الفترة الاختبارية من صنف ب (8) وألا يقل رصيد النقط المخصصلرخصته عن إثنتي عشرة نقطة. للحصول على رخصة السياقة من الصنفين ج (0) ود (0) ؛
- رخصة السياقة بعد انتهاء الفترة الاختبارية من صنف ب (8) وأ
📚 Sources: ['المادة 152', 'المادة 37', 'المادة 132']


## 7. Comparaison des 3 LLMs

In [27]:
import time

test_questions = [
    "ما هي شروط الحصول على رخصة السياقة؟",
    "ما هو الحد الأقصى للسرعة داخل المدن؟",
    "ما هي عقوبة القيادة تحت تأثير الكحول؟"
]

comparison_results = []

print("=" * 80)
print("📊 COMPARAISON DES 3 LLMs")
print("=" * 80)

for question in test_questions:
    print(f"\n{'─' * 60}")
    print(f"❓ Question: {question}")
    print('─' * 60)

    for model_key in LLM_MODELS.keys():
        start = time.time()
        result = generate_answer(question, model_key=model_key, k=4)
        elapsed = time.time() - start

        comparison_results.append({
            "question": question,
            "model": model_key,
            "answer_preview": result["answer"][:150],
            "latency_s": round(elapsed, 2),
            "out_of_domain": result["out_of_domain"],
            "num_sources": len(result["sources"])
        })

        print(f"\n🤖 [{model_key}] ({elapsed:.2f}s)")
        print(f"   Réponse: {result['answer'][:200]}...")
        print(f"   Sources: {len(result['sources'])} articles")

# Tableau comparatif
print("\n" + "=" * 80)
print("📈 TABLEAU COMPARATIF DES PERFORMANCES")
print("=" * 80)
pd.DataFrame(comparison_results)

📊 COMPARAISON DES 3 LLMs

────────────────────────────────────────────────────────────
❓ Question: ما هي شروط الحصول على رخصة السياقة؟
────────────────────────────────────────────────────────────

🤖 [LLaMA-3.3-70B] (1.59s)
   Réponse: تحدد شروط الحصول على رخصة السياقة في القانون 52.05، ويمكن تلخيصها على النحو التالي: 
- يجب أن يكون الشخص حاصلاً على رخصة السياقة بعد انتهاء الفترة الاختبارية من صنف ب (8) وألا يقل رصيد النقط المخصص لر...
   Sources: 3 articles

🤖 [LLaMA-3.1-8B] (0.53s)
   Réponse: حسب المادة 251، شروط الحصول على رخصة السياقة هي:

- أن يكون حاصلا على رخصة السياقة بعد انتهاء الفترة الاختبارية من صنف ب (8) وألا يقل رصيد النقط المخصصلرخصته عن إثنتي عشرة نقطة. للحصول على رخصة السياق...
   Sources: 3 articles

────────────────────────────────────────────────────────────
❓ Question: ما هو الحد الأقصى للسرعة داخل المدن؟
────────────────────────────────────────────────────────────

🤖 [LLaMA-3.3-70B] (0.64s)
   Réponse: لا يوجد في السياق المحدد الحد الأقصى للسرعة داخل المدن، ولكن ي

,question,model,answer_preview,latency_s,out_of_domain,num_sources
0,ما هي شروط الحصول على رخصة السياقة؟,LLaMA-3.3-70B,تحدد شروط الحصول على رخصة السياقة في القانون 5...,1.59,False,3
1,ما هي شروط الحصول على رخصة السياقة؟,LLaMA-3.1-8B,حسب المادة 251، شروط الحصول على رخصة السياقة ه...,0.53,False,3
2,ما هو الحد الأقصى للسرعة داخل المدن؟,LLaMA-3.3-70B,لا يوجد في السياق المحدد الحد الأقصى للسرعة دا...,0.64,False,3
3,ما هو الحد الأقصى للسرعة داخل المدن؟,LLaMA-3.1-8B,حسناً. الإجابة هي: 50 كيلومترا في الساعة (الما...,0.24,False,3
4,ما هي عقوبة القيادة تحت تأثير الكحول؟,LLaMA-3.3-70B,تتراوح العقوبة بين الحبس من شهر واحد إلى سنتين...,0.55,False,3
5,ما هي عقوبة القيادة تحت تأثير الكحول؟,LLaMA-3.1-8B,المادة 172.\n\nيعاقب بالحبس من ثلاثة (3) أشهر ...,0.23,False,3


## 8. Évaluation du Retriever (Precision & Recall)

In [28]:
# Ground truth basé sur les articles du code
eval_dataset = [
    {"question": "ما هي شروط الحصول على رخصة السياقة؟",
     "keywords": ["رخصة", "سياقة", "شروط", "امتحان", "فحص طبي"],
     "expected_articles": [1, 10, 11, 12]},

    {"question": "ما هو الحد الأقصى للسرعة داخل المدن؟",
     "keywords": ["سرعة", "حد", "أقصى", "مدن", "كيلومتر"],
     "expected_articles": [53, 54]},

    {"question": "ما هي عقوبة القيادة تحت تأثير الكحول؟",
     "keywords": ["كحول", "عقوبة", "سكر", "حبس", "غرامة"],
     "expected_articles": [183, 207]},
]

def evaluate_retrieval_semantic(eval_data, k=5):
    """
    Évaluation sémantique du retriever basée sur les mots-clés.
    """
    precisions = []
    recalls = []

    for sample in eval_data:
        query = sample["question"]
        keywords = set(sample["keywords"])

        retrieved_docs = retrieve(query, k=k, min_score=0.2)

        # Vérifier si les documents contiennent les mots-clés
        relevant_retrieved = 0
        for doc in retrieved_docs:
            doc_text = doc["texte"]
            if any(kw in doc_text for kw in keywords):
                relevant_retrieved += 1

        precision = relevant_retrieved / k if k > 0 else 0
        recall = relevant_retrieved / len(keywords) if keywords else 0

        precisions.append(precision)
        recalls.append(recall)

        print(f"\n📝 {query[:50]}...")
        print(f"   Documents trouvés: {len(retrieved_docs)}")
        print(f"   Documents pertinents: {relevant_retrieved}/{k}")
        print(f"   Precision@{k}: {precision:.2f}")
        print(f"   Recall: {recall:.2f}")

    avg_precision = sum(precisions) / len(precisions) if precisions else 0
    avg_recall = sum(recalls) / len(recalls) if recalls else 0
    f1 = 2 * avg_precision * avg_recall / (avg_precision + avg_recall + 1e-9)

    print("\n" + "=" * 60)
    print("📊 RÉSULTATS DE L'ÉVALUATION")
    print("=" * 60)
    print(f"📈 Precision moyenne@{k}: {avg_precision:.3f}")
    print(f"📈 Recall moyen: {avg_recall:.3f}")
    print(f"🎯 F1-score: {f1:.3f}")

    return {"precision": avg_precision, "recall": avg_recall, "f1": f1}

print("🔍 ÉVALUATION DU RETRIEVER")
print("=" * 60)
metrics = evaluate_retrieval_semantic(eval_dataset, k=5)

🔍 ÉVALUATION DU RETRIEVER

📝 ما هي شروط الحصول على رخصة السياقة؟...
   Documents trouvés: 5
   Documents pertinents: 5/5
   Precision@5: 1.00
   Recall: 1.00

📝 ما هو الحد الأقصى للسرعة داخل المدن؟...
   Documents trouvés: 5
   Documents pertinents: 5/5
   Precision@5: 1.00
   Recall: 1.00

📝 ما هي عقوبة القيادة تحت تأثير الكحول؟...
   Documents trouvés: 5
   Documents pertinents: 4/5
   Precision@5: 0.80
   Recall: 0.80

📊 RÉSULTATS DE L'ÉVALUATION
📈 Precision moyenne@5: 0.933
📈 Recall moyen: 0.933
🎯 F1-score: 0.933


## 9. Détection des questions hors domaine

In [29]:
# Questions hors domaine (ne doivent PAS être traitées)
out_of_domain_questions = [
    "ما هو عاصمة فرنسا؟",
    "كيف أطبخ الكسكس؟",
    "ما هو سعر الذهب اليوم؟",
    "من هو رئيس الحكومة المغربية؟",
]

# Questions dans le domaine (DOIVENT être traitées)
in_domain_questions = [
    "ما هي شروط رخصة السياقة؟",
    "ما هو الحد الأقصى للسرعة؟",
    "ما هي عقوبة عدم ارتداء حزام الأمان؟",
]

print("🔍 TEST DE DÉTECTION HORS DOMAINE")
print("=" * 60)

print("\n📌 Questions HORS domaine (devraient être rejetées):")
tp_ood = 0
for q in out_of_domain_questions:
    is_ood = is_out_of_domain(q)
    status = "✅ REJETÉ" if is_ood else "❌ ACCEPTÉ (faux négatif)"
    if is_ood:
        tp_ood += 1
    print(f"   {status}: {q}")

print("\n📌 Questions DANS le domaine (devraient être acceptées):")
tn_ood = 0
for q in in_domain_questions:
    is_ood = is_out_of_domain(q)
    status = "✅ ACCEPTÉ" if not is_ood else "❌ REJETÉ (faux positif)"
    if not is_ood:
        tn_ood += 1
    print(f"   {status}: {q}")

accuracy = (tp_ood + tn_ood) / (len(out_of_domain_questions) + len(in_domain_questions))
print("\n" + "=" * 60)
print(f"📊 PRÉCISION DE LA DÉTECTION: {accuracy:.0%}")
print(f"   ✓ Vrais positifs (hors domaine rejetés): {tp_ood}/{len(out_of_domain_questions)}")
print(f"   ✓ Vrais négatifs (domaine acceptés): {tn_ood}/{len(in_domain_questions)}")

🔍 TEST DE DÉTECTION HORS DOMAINE

📌 Questions HORS domaine (devraient être rejetées):
   ✅ REJETÉ: ما هو عاصمة فرنسا؟
   ✅ REJETÉ: كيف أطبخ الكسكس؟
   ❌ ACCEPTÉ (faux négatif): ما هو سعر الذهب اليوم؟
   ❌ ACCEPTÉ (faux négatif): من هو رئيس الحكومة المغربية؟

📌 Questions DANS le domaine (devraient être acceptées):
   ✅ ACCEPTÉ: ما هي شروط رخصة السياقة؟
   ✅ ACCEPTÉ: ما هو الحد الأقصى للسرعة؟
   ✅ ACCEPTÉ: ما هي عقوبة عدم ارتداء حزام الأمان؟

📊 PRÉCISION DE LA DÉTECTION: 71%
   ✓ Vrais positifs (hors domaine rejetés): 2/4
   ✓ Vrais négatifs (domaine acceptés): 3/3


## 10. Interface utilisateur (Gradio)

In [31]:
import gradio as gr

def chat_interface(question, model_choice, num_docs):
    """Interface Gradio pour le chatbot RAG."""
    if not question.strip():
        return "الرجاء إدخال سؤال.", ""

    result = generate_answer(question, model_key=model_choice, k=int(num_docs))

    # Formatage des sources
    if result["out_of_domain"]:
        sources_text = "⚠️ Cette question semble hors du domaine du code de la route."
    else:
        sources_lines = []
        for src in result["sources"]:
            sources_lines.append(f"### 📖 {src['article']}\n{src['texte'][:300]}...")
        sources_text = "\n\n---\n\n".join(sources_lines) if sources_lines else "Aucune source disponible."

    return result["answer"], sources_text

def get_model_info():
    """Retourne les informations sur les modèles réellement disponibles."""
    return """
    ### 🤖 Modèles disponibles:

    - **LLaMA-3.3-70B** ⭐
      Meilleure qualité, recommandé pour les réponses juridiques complexes

    - **LLaMA-3.1-8B** ⚡
      Rapide, bon compromis performance / vitesse
    """

with gr.Blocks(title="نظام RAG - مدونة السير على الطرق", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🚗 نظام RAG لمدونة السير على الطرق المغربية (القانون 52.05)

    اطرح أسئلتك حول قانون السير المغربي وسيجيب النظام بناءً على النصوص القانونية الرسمية.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(
                label="💬 سؤالك",
                placeholder="مثال: ما هي شروط الحصول على رخصة السياقة؟",
                lines=3
            )
            with gr.Row():
                model_dropdown = gr.Dropdown(
                    choices=list(LLM_MODELS.keys()),
                    value="LLaMA-3.1-8B",
                    label="🧠 نموذج اللغة"
                )
                num_docs_slider = gr.Slider(
                    minimum=2, maximum=8, value=4, step=1,
                    label="📚 عدد المقتطفات المسترجعة"
                )
            submit_btn = gr.Button("🔍 البحث والإجابة", variant="primary", size="lg")

            gr.Markdown(get_model_info())

        with gr.Column(scale=2):
            answer_output = gr.Textbox(
                label="📝 الجواب",
                lines=10,
                show_copy_button=True
            )
            sources_output = gr.Markdown(label="📚 المصادر القانونية")

    submit_btn.click(
        fn=chat_interface,
        inputs=[question_input, model_dropdown, num_docs_slider],
        outputs=[answer_output, sources_output]
    )

    # Exemples
    gr.Markdown("### 📌 Exemples de questions")
    gr.Examples(
        examples=[
            ["ما هي شروط الحصول على رخصة السياقة؟", "LLaMA-3.1-8B", 4],
            ["ما هو الحد الأقصى للسرعة على الطريق السيار؟", "Qwen-2.5-7B", 4],
            ["ما هي عقوبة السياقة تحت تأثير الكحول؟", "LLaMA-3.1-70B", 4],
            ["ما هي قواعد استخدام الهاتف أثناء القيادة؟", "LLaMA-3.1-8B", 3],
            ["كم عدد النقط التي تفقدها رخصة السياقة عند تجاوز السرعة؟", "Qwen-2.5-7B", 3],
        ],
        inputs=[question_input, model_dropdown, num_docs_slider]
    )

print("🚀 Lancement de l'interface Gradio...")
demo.launch(share=True, debug=False)

/tmp/ipykernel_11697/2933999547.py:33: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="نظام RAG - مدونة السير على الطرق", theme=gr.themes.Soft()) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: Qwen-2.5-7B or set allow_custom_value=True.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/gradio/components/dropdown.py:230: UserWarning: The value passed into gr.Dropdown() is not in the list of choices. Please update the list of choices to include: LLaMA-3.1-70B or set allow_custom_value=True.
  warnings.warn(


🚀 Lancement de l'interface Gradio...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://46bd9380208672c995.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 11. Rapport technique

### Architecture du système RAG

| Composant | Technologie | Justification |
|-----------|-------------|---------------|
| Extraction PDF | Tesseract OCR + pdf2image | Support de l'arabe, configuration optimisée |
| Modèle d'embedding | paraphrase-multilingual-MiniLM-L12-v2 | Multilingue, bon pour l'arabe |
| Base vectorielle | FAISS (IndexFlatIP) | Rapide, similarité cosinus |
| LLM 1 | LLaMA-3.1-70B (Groq) | Haute performance |
| LLM 2 | LLaMA-3.1-8B (Groq) | Rapide, économique |
| LLM 3 | Qwen-2.5-7B (Groq) | Excellent en arabe |
| Interface | Gradio | Simple, déployable |

### Pipeline du système

```
Question utilisateur
       │
       ▼
┌──────────────────────────────────────┐
│ 1. Détection hors domaine            │
│    - Seuil de similarité: 0.25       │
└──────────────────────────────────────┘
       │
       ▼
┌──────────────────────────────────────┐
│ 2. Recherche vectorielle (FAISS)     │
│    - Top-k chunks (k=4-5)            │
│    - Similarité cosinus              │
└──────────────────────────────────────┘
       │
       ▼
┌──────────────────────────────────────┐
│ 3. Construction du prompt            │
│    - Contexte légal                   │
│    - Instructions pour le LLM        │
└──────────────────────────────────────┘
       │
       ▼
┌──────────────────────────────────────┐
│ 4. Génération (LLM via Groq)         │
│    - Température: 0.2                │
│    - Max tokens: 512                 │
└──────────────────────────────────────┘
       │
       ▼
    Réponse + Sources
```

### Résultats et métriques

| Métrique | Valeur |
|----------|--------|
| Articles extraits | {len(df)} |
| Chunks créés | {len(df_chunks)} |
| Taille vecteurs | {dimension} |
| Seuil hors domaine | 0.25 |

### Améliorations implémentées

1. **OCR optimisé**: Configuration spécifique pour l'arabe avec whitelist de caractères
2. **Chunking intelligent**: Découpage basé sur les phrases complètes
3. **Détection hors domaine**: Basée sur le seuil de similarité
4. **Comparaison de 3 LLMs**: Évaluation des performances

### Limitations

- Qualité de l'OCR peut affecter la pertinence des résultats
- Les amendements récents (post-2024) ne sont pas inclus
- Dépendance à l'API Groq (clé API requise)

### Perspectives d'amélioration

- Utiliser un modèle d'embedding spécialisé en arabe juridique
- Ajouter un reranker (Cross-Encoder) pour améliorer la précision
- Implémenter un cache pour les questions fréquentes
- Ajouter la possibilité de mettre à jour la base documentaire